## Objective

Train a Linear Regression model using bootstrap to predict the best places for 
new oil wells.

Choose the region with the highest total profit for the selected oil wells.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from numpy.random import RandomState

## Business parameters

In [3]:
# all values are in dolars

# total cost to build 200 wells
total_cost = 100000000

# cost per well
well_cost = total_cost / 200

# Profit per barrel is $4.5. Reserves volume unit is in thousands of barrels.
# profit per unit on 'product' column [$/un]
unit_profit = 4500

# break-even voloume per well
break_even_volume = well_cost / unit_profit

## Functions

In [4]:
def model_train(df):
    features = df.drop('product', axis=1)
    target = df['product']

    features_train, features_val, target_train, target_val = train_test_split(
        features,
        target,
        test_size=0.25,
        random_state=123,
    )

    scaler = StandardScaler()
    features_train = scaler.fit_transform(features_train)
    features_val = scaler.transform(features_val)

    model = LinearRegression()
    model.fit(features_train, target_train)
    predict_val = model.predict(features_val)

    predict_val_mean = predict_val.mean()
    rmse = mean_squared_error(target_val, predict_val) ** 0.5

    print(
        f'Mean predicted value: {predict_val_mean:.3f}\n'
        f'Root Mean Squared Error: {rmse:.3f}'
    )

    train_score = model.score(features_train, target_train)
    val_score = model.score(features_val, target_val)

    print(f'Train score: {train_score:.3f}')
    print(f'Validation score: {val_score:.3f}')

    return predict_val, target_val

def profit_estimate(predict_val, target_val):

    rand_gen = RandomState(123)

    val_df = pd.DataFrame({'predict': predict_val, 'target': target_val})

    # real profit values
    val_df['profit'] = (val_df['target'] * unit_profit) - well_cost

    profits_list = []
    for i in range(1000):
        subsample = val_df.sample(500, replace=True, random_state = rand_gen)

        # select wells by predicted values
        subsample = subsample.sort_values(by='predict', ascending=False)
        top_wells = subsample.head(200)

        # calculate profit with real values
        sample_profit = top_wells['profit'].sum()
        profits_list.append(sample_profit)

    profits = pd.Series(profits_list)
    profit_mean = profits.mean()
    profit_std = profits.std()
    
    lower_quantile = profits.quantile(0.025)
    upper_quantile = profits.quantile(0.975)

    loss_risk = (profits < 0).mean()

    print(f'Mean profit: {profit_mean:,.2f}')
    print(f'Standard Deviation: {profit_std:,.2f}')
    print(f'95% confidence interval: {lower_quantile:,.2f}; {upper_quantile:,.2f}')

    print(f'Loss risk: {loss_risk:.3%}')

## Read files
There is a csv for each of three regions.

In [5]:
# dictionary to store other results later
df_dict = {
    'df0': {'df': pd.read_csv('./datasets/geo_data_0.csv')},
    'df1': {'df': pd.read_csv('./datasets/geo_data_1.csv')},
    'df2': {'df': pd.read_csv('./datasets/geo_data_2.csv')}
}

## Data Cleaning

In [6]:
for value in df_dict.values():
    print(value['df'].head())

      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647
      id         f0         f1        f2     product
0  kBEdx -15.001348  -8.276000 -0.005876    3.179103
1  62mP7  14.272088  -3.475083  0.999183   26.953261
2  vyE1P   6.263187  -5.948386  5.001160  134.766305
3  KcrkZ -13.081196 -11.506057  4.999415  137.945408
4  AHL4O  12.702195  -8.147433  5.004363  134.766305
      id        f0        f1        f2     product
0  fwXo0 -1.146987  0.963328 -0.828965   27.758673
1  WJtFt  0.262778  0.269839 -2.530187   56.069697
2  ovLUW  0.194587  0.289035 -5.586433   62.871910
3  q6cA6  2.236060 -0.553760  0.930038  114.572842
4  WPMUX -0.515993  1.716266  5.899011  149.600746


f0, f1, f2 — locations significative characteristics.

product — reserves volume (thousands of barrels).

In [7]:
for value in df_dict.values():
    value['df'].info()


<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   -----------

There is no null values.

In [8]:
for value in df_dict.values():
    print(value['df']['id'].nunique())


99990
99996
99996


In [9]:
for value in df_dict.values():
    df = value['df']
    print(df[df['id'].duplicated(keep=False)].sort_values('id'))


          id        f0        f1         f2     product
66136  74z30  1.084962 -0.312358   6.990771  127.643327
64022  74z30  0.741456  0.459229   5.153109  140.771492
51970  A5aEY -0.180335  0.935548  -2.094773   33.020205
3389   A5aEY -0.039949  0.156872   0.209861   89.249364
69163  AGS9W -0.933795  0.116194  -3.655896   19.230453
42529  AGS9W  1.454747 -0.479651   0.683380  126.370504
931    HZww2  0.755284  0.368511   1.863211   30.681774
7530   HZww2  1.061194 -0.373969  10.430210  158.828695
63593  QcMuo  0.635635 -0.473422   0.862670   64.578675
1949   QcMuo  0.506563 -0.323775  -2.215583   75.496502
75715  Tdehs  0.112079  0.430296   3.218993   60.964018
21426  Tdehs  0.829407  0.298807  -0.049563   96.035308
92341  TtcGQ  0.110711  1.022689   0.911381  101.318008
60140  TtcGQ  0.569276 -0.104876   6.440215   85.350186
89582  bsk9y  0.398908 -0.400253  10.122376  163.433078
97785  bsk9y  0.378429  0.005837   0.160827  160.637302
41724  bxg6G -0.823752  0.546319   3.630479   93

The duplicated values will be discarded, because there is no indication of 
which is the correct one. In a real situation, this would be verified with the 
team responsible for the data.

In [10]:
for value in df_dict.values():
    value['df'] = value['df'].drop_duplicates(subset='id', keep=False, ignore_index=True)

The column 'id' is irrelevant for model training and will be discarded.

In [11]:
for value in df_dict.values():
    value['df'] = value['df'].drop('id', axis=1)

In [ ]:
print(f'Break-even barrels quantity: {break_even_volume:.3f}')

for key, value in df_dict.items():
    mean_reserve = value['df']['product'].mean()
    condition = 'below' if mean_reserve < break_even_volume else 'above'

    print(f'{key} mean reserve: {mean_reserve:.3f} ({condition} break-even)')

Break-even barrels quantity: 111.111
df0 mean reserve: 92.499 (below break-even
df1 mean reserve: 68.824 (below break-even
df2 mean reserve: 94.999 (below break-even


No region has a mean reserve value above the break-even point. The wells must 
be selected to increase this value.

## Model training

In [13]:
for key, value in df_dict.items():
    print(key)
    value['predict_val'], value['target_val'] = model_train(value['df'])
    print()

df0
Mean predicted value: 92.734
Root Mean Squared Error: 37.556
Train score: 0.276
Validation score: 0.276

df1
Mean predicted value: 68.610
Root Mean Squared Error: 0.894
Train score: 1.000
Validation score: 1.000

df2
Mean predicted value: 95.002
Root Mean Squared Error: 39.941
Train score: 0.198
Validation score: 0.201



The model for df1 has a much lower RMSE and much higher R². The problem is that this 
perfect R² can be a data leakeage problem. Verify the correlations to see if 
there is some clue.

In [14]:
for key, value in df_dict.items():
    print(key)
    print(value['df'].corr())

df0
               f0        f1        f2   product
f0       1.000000 -0.440724 -0.003204  0.143504
f1      -0.440724  1.000000  0.001783 -0.192338
f2      -0.003204  0.001783  1.000000  0.483628
product  0.143504 -0.192338  0.483628  1.000000
df1


               f0        f1        f2   product
f0       1.000000  0.182263 -0.001821 -0.030534
f1       0.182263  1.000000 -0.002608 -0.010167
f2      -0.001821 -0.002608  1.000000  0.999397
product -0.030534 -0.010167  0.999397  1.000000
df2
               f0        f1        f2   product
f0       1.000000  0.000501 -0.000454 -0.001974
f1       0.000501  1.000000  0.000763 -0.001046
f2      -0.000454  0.000763  1.000000  0.445873
product -0.001974 -0.001046  0.445873  1.000000


In df1 the correlation between f2 and product is very close to 1, which does 
not happen at the other datasets.

f2 can be a variable related directly to the well production, instead of 
geological data. This need to be verified with the team responsible for data 
acquisition.

## Profit estimate

### Profit estimate with all models.

In [15]:
for key, value in df_dict.items():
    print(key)
    profit_estimate(value['predict_val'], value['target_val'])
    print()

df0
Mean profit: 4,348,355.45
Standard Deviation: 2,738,971.63
95% confidence interval: -1,264,032.08; 9,413,923.70
Loss risk: 6.100%

df1
Mean profit: 4,281,476.72
Standard Deviation: 2,032,016.60
95% confidence interval: 360,488.18; 8,416,097.43
Loss risk: 1.200%

df2
Mean profit: 3,752,141.80
Standard Deviation: 2,652,102.50
95% confidence interval: -1,335,913.40; 9,028,936.77
Loss risk: 7.000%



The region represented by df1 is the only with the loss risk below 2.5%, which 
is the maximum acceptable for this project.

If the evidence for data leakege was discarded when verified by the team 
responsible for the data, region from df1 would be recommended. As this was not
solved before the model training step, no region can be recommended. 